In [1]:
import duckdb


### Add Kentucky Boundary

Add Kentucky Boundary from _kybnd.parquet_ to latest.ddb main tables

I will be using `with duckdb.connect()` in order to have connections close when queries are complete.  
This will allow me to test **latest.ddb** in DBeaver UI.

In [2]:
with duckdb.connect('../latest.ddb') as conn:
    conn.sql("""
             INSTALL spatial;
             LOAD spatial;

             INSTALL aws;

             CREATE OR REPLACE TABLE kybnd AS
             SELECT * 
             FROM read_parquet('../parquets/kybnd.parquet')
             """)
    
    print('Table ky_bnd created.')

Table ky_bnd created.


In [55]:
# Tables where Kentucky can be queried from the table
# eliminating the need to run a spatial query on Ky's 
# border geometry.
# bbox will be used to narrow down the field of view.


with duckdb.connect('../latest.ddb') as conn:
  conn.sql("""
  LOAD spatial;
  LOAD aws;

  CALL load_aws_credentials();
  
  CREATE OR REPLACE TABLE places as
    SELECT 	p.id,
            p.version,
            p.names.primary as name_primary,
            p.names.common as name_common,
            p.names.rules as name_rules,
            p.sources[1].dataset as src1_dataset,
            p.sources[2].dataset as src2_dataset,
            p.sources[1].license as src1_license,
            p.sources[2].license as src2_license,
            p.sources[1].record_id as src1_record_id,
            p.sources[1].update_time as src1_update_time,
            p.sources[2].update_time as src2_update_time,
            p.sources[1].confidence as src1_confidence,
            p.sources[1].between as src1_between,	
            p.categories.primary as category_primary,
            p.categories.alternate[1] as category_alternate1,
            p.categories.alternate[2] as category_alternate2,
            basic_category,
            p.taxonomy.primary as taxonomy_primary,
            p.taxonomy.hierarchy[1] as taxonomy_hierarchy1,
            p.taxonomy.hierarchy[2] as taxonomy_hierarchy2,
            p.taxonomy.hierarchy[3] as taxonomy_hierarchy3,
            p.taxonomy.hierarchy[4] as taxonomy_hierarchy4,
            p.taxonomy.hierarchy[5] as taxonomy_hierarchy5,
            p.taxonomy.hierarchy[6] as taxonomy_hierarchy6,
            p.taxonomy.alternates as taxonomy_alternate,
            p.confidence,
            p.websites,
            p.socials,
            p.emails,
            p.phones,
            p.brand.wikidata as brand_wikidata,
            p.brand.names.primary as brand_names,
            p.addresses[1].freeform as address_freeform,
            p.addresses[1].locality as locality,
            p.addresses[1].postcode as postcode,
            p.addresses[1].region as region,
            p.addresses[1].country as country,
            p.operating_status,
            p.geometry 
        FROM place_vw as p
        WHERE p.bbox.xmin BETWEEN -89.57122 AND -81.96479
          AND p.bbox.ymin BETWEEN 36.49706 AND 39.14774
          AND ST_Intersects(
              p.geometry,
              (SELECT geometry FROM kybnd)
              );

        COPY (
           SELECT * from places
           ) TO 's3://kyvector/overture-maps/ky_overturemaps_places.parquet'
           WITH (FORMAT 'parquet');                
""")

  print(f'Table place created.\n'
        f'Parquet file upload to s3://kyvector/overture-maps/')

Table place created.
Parquet file upload to s3://kyvector/overture-maps/


### Clip to Kentucky Border

These tables will need further processsing.  

 - Step 1: query to bounding box
 - Step2:  clip to Kentucky boundary

In [54]:
with duckdb.connect('../latest.ddb') as conn:
    conn.sql("""
        LOAD spatial;
        LOAD aws;

        CALL load_aws_credentials();
        

        CREATE OR REPLACE TABLE infrastructure as
            SELECT  i.id,
                i.names.primary,
                i.level,
                i.subtype,
                i.class,
                i.height,
                i.surface,
                i.sources[1].property as src_property,
                i.sources[1].dataset as src_dataset,
                i.sources[1].license as src_license,
                i.sources[1].update_time as src_update_time,
                i.source_tags,
                i.wikidata,
                i.geometry
            FROM infrastructure_vw as i
            WHERE i.bbox.xmin BETWEEN -89.6 AND -82
            AND i.bbox.ymin BETWEEN 36.4 AND 39.2
            AND ST_Intersects(
                    i.geometry,
                    (SELECT geometry FROM kybnd));
        
        COPY (
             SELECT * FROM infrastructure
             ) TO 's3://kyvector/overture-maps/ky_overturemaps_infrastructure.parquet'
             WITH (FORMAT 'parquet')
""")
    
    print(f'Table infrastructure created\n'
          f'Parquet file uploaded so s3://kyvector/overture-maps/')

Table infrastructure created
Parquet file uploaded so s3://kyvector/overture-maps


### Buildings

In [6]:
with duckdb.connect('../latest.ddb') as conn:
    conn.sql("""
            LOAD spatial;

            CREATE OR REPLACE
            TABLE buildings as
            SELECT
                id,
                    version,
                    sources[1].dataset as source,
                    names.primary as name,
                    level,
                    subtype,
                    height,
                --		has_parts,
                --		is_underground,
                        num_floors,
                --		num_floors_underground,
                --		min_height,
                --		min_floor,
                --		facade_color,
                --		facade_material,
                    roof_material,
                    roof_shape,
                --		roof_direction,
                    roof_color,
                    roof_height,
            --		bbox,
                    geometry
            FROM
                building_vw
            WHERE
                bbox.xmin BETWEEN -89.57122 AND -81.96479
                AND bbox.ymin BETWEEN 36.49706 AND 39.14774
                AND ST_Intersects(geometry,
                        (SELECT geometry FROM kybnd));
             """)
    
    print('Table buildings created.')